# ReWild all-hybrid — local MPS SFT + GRPO
Use the official hybrid JSON files and existing trajectory thoughts. No API key. Default local profile samples all available types; full profile evaluates all valid official test questions. Thought excerpts are not independently verified. Start with smoke. Run cells in order in a fresh kernel.

In [1]:
# Optional: install in your MPS environment, then restart kernel.
# %pip install "transformers==4.52.4" "peft==0.15.2" "accelerate>=1.6,<2" "huggingface_hub>=0.30,<1"

In [2]:
# Set profile before running config; change to local after smoke, then restart kernel.
import os
os.environ["REWILD_PROFILE"] = "smoke"

In [3]:
# # ReWild all-hybrid — local MPS reasoning SFT + GRPO


import os
# Set before importing torch; allows unsupported MPS ops to fall back to CPU.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
import gc, json, random, re, math, hashlib
from collections import Counter, defaultdict
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    torch.cuda.manual_seed_all(SEED)
else:
    raise RuntimeError('No MPS or CUDA device available. Check your PyTorch installation.')
DTYPE = torch.float32  # stable default on MPS; memory use is higher than FP16
PROFILE = os.getenv('REWILD_PROFILE', 'local')
assert PROFILE in ('local','smoke','full')
SMOKE = PROFILE == 'local'
def setting(name, default): return int(os.getenv('REWILD_'+name, str(default)))
MODEL_ID = 'Qwen/Qwen3-0.6B-Base'
SFT_EXAMPLES = setting('SFT_EXAMPLES', 8 if SMOKE else (400 if PROFILE=='full' else 100))
VAL_EXAMPLES = setting('VAL_EXAMPLES', 4 if SMOKE else 12)
GRPO_STEPS = setting('GRPO_STEPS', 2 if SMOKE else (60 if PROFILE=='full' else 12))
GRPO_ROLLOUTS = setting('GRPO_ROLLOUTS', 2 if SMOKE else 4)
TEST_EXAMPLES = setting('TEST_EXAMPLES', 4 if SMOKE else (0 if PROFILE=='full' else 40))
SFT_EPOCHS = setting('SFT_EPOCHS', 1 if SMOKE else 2)
MAX_SEQ_LEN = setting('MAX_SEQ_LEN', 1024)
EVAL_MAX_NEW_TOKENS = 500
GRPO_MAX_NEW_TOKENS = 500
SFT_BATCH = 2
GRAD_ACCUM = 2
EVAL_BATCH = 2
SFT_LR = 1e-4
GRPO_LR = 5e-6
GRPO_CLIP_EPS = 0.2
GRPO_OPT_EPOCHS = 1
GRPO_KL_BETA = 0.02
GRPO_VAL_EVERY = 6
assert min(SFT_EXAMPLES,VAL_EXAMPLES,GRPO_STEPS,SFT_EPOCHS,SFT_BATCH,GRAD_ACCUM,EVAL_BATCH)>0
assert GRPO_ROLLOUTS>=2
OUT = Path(os.getenv('REWILD_OUT', f'rewild_mps_{PROFILE}'))
OUT.mkdir(parents=True, exist_ok=True)

def empty_cache():
    gc.collect()
    if device.type=='mps': torch.mps.empty_cache()
    elif device.type=='cuda': torch.cuda.empty_cache()

def save_json(name,obj):
    dest=OUT/name; tmp=dest.with_suffix(dest.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str,ensure_ascii=False),encoding='utf-8'); tmp.replace(dest)
    return dest

print(torch.__version__,device,DTYPE,'batch',SFT_BATCH,'effective',SFT_BATCH*GRAD_ACCUM)
print('Run profile:',PROFILE,'output:',OUT.resolve())

/opt/anaconda3/envs/rewild_mps/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.6.0 mps torch.float32 batch 2 effective 4
Run profile: smoke output: /Users/pritishinde/reasoning-from-scratch/ch06/01_main-chapter-code/new_code/rewild_mps_smoke


## 1. Load official hybrid data and reuse dataset reasoning

In [4]:

PROFILE = "local"
SMOKE = False

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
DTYPE = torch.float32

# Dataset
SFT_EXAMPLES = 300
VAL_EXAMPLES = 50
TEST_EXAMPLES = 50

# SFT
SFT_EPOCHS = 2
SFT_BATCH = 2
GRAD_ACCUM = 4
SFT_LR = 5e-5
MAX_SEQ_LEN = 2048

# Generation
EVAL_BATCH = 2
EVAL_MAX_NEW_TOKENS = 512
GRPO_MAX_NEW_TOKENS = 512

# GRPO
GRPO_STEPS = 100
GRPO_ROLLOUTS = 4
GRPO_LR = 3e-5 
GRPO_CLIP_EPS = 0.2
GRPO_OPT_EPOCHS = 1
GRPO_KL_BETA = 0.05
GRPO_VAL_EVERY = 10

# Save separately from previous experiment
OUT = Path("rewild_mps_local_v2")
OUT.mkdir(parents=True, exist_ok=True)


In [5]:
import os
locations = [
    Path(os.environ['REWILD_DATA_DIR']).expanduser() if os.environ.get('REWILD_DATA_DIR') else None,
    Path.cwd().parent / 'rewild_data',
    Path.cwd() / 'rewild_data',
    Path.cwd(),
]
DATA_DIR = next((p for p in locations if p is not None
                 and (p / 'hybrid_train.json').exists()
                 and (p / 'hybrid_test.json').exists()), None)
if DATA_DIR is None:
    DATA_DIR = Path(hf_hub_download('yuan-yang/ReWild','hybrid_train.json',repo_type='dataset')).parent
    hf_hub_download('yuan-yang/ReWild','hybrid_test.json',repo_type='dataset')
with (DATA_DIR / 'hybrid_train.json').open(encoding='utf-8') as f:
    train_raw = json.load(f)
with (DATA_DIR / 'hybrid_test.json').open(encoding='utf-8') as f:
    test_raw = json.load(f)
assert isinstance(train_raw, list) and isinstance(test_raw, list)
print('[DATA] Folder:', DATA_DIR)
print('[DATA] Train records:', len(train_raw), '| test records:', len(test_raw))

def build_prompt(problem):
    return ("### Problem\n" + problem.strip() + "\n\n### Instructions\n"
            "Choose the one correct statement. Let's think step by step. "
            "Explain your reasoning briefly inside <reasoning>...</reasoning>. "
            "Then write Final answer: followed by the integer number of the correct statement.\n\n"
            "### Solution\n")


def build_item(raw):
    question=raw.get('input'); options=raw.get('options') or []
    gold=str(raw.get('label','')).strip()
    if not isinstance(question,str) or not question.strip() or not gold.isdecimal() or not 1<=int(gold)<=len(options):
        raise ValueError('Missing input or invalid option label')
    return dict(question=question,prompt=build_prompt(question),ground_truth=str(int(gold)),
                n_options=len(options),qtype=str(raw.get('qtype','unknown')))


def build_reasoning(raw, item):
    """Reuse existing trajectory thought text, not generated/fabricated explanations.
    These are tool-guided references; thought-only text may omit tool evidence.
    Return None if no usable concise trace. No test trace is read.
    """
    trajectory=raw.get('hyb_syn_traj') or {}
    blocks=[]
    for entry in trajectory.get('exp_ls') or []:
        response=entry.get('response') or {}
        thought=response.get('thought')
        if not isinstance(thought,str): continue
        thought=thought.strip()
        if not thought or thought.lower() in ('none','null','n/a'): continue
        # Do not include an explicit conflicting final answer in an SFT target.
        explicit=re.findall(r'(?im)final\s+answer\s*[:=]\s*(?:N\s*=\s*)?(?:option\s*|statement\s*)?(\d+)\b',thought)
        if any(str(int(x))!=item['ground_truth'] for x in explicit): return None
        thought=re.sub(r'</?reasoning>', '', thought, flags=re.I).strip()
        if thought and thought not in blocks: blocks.append(thought)
    if not blocks: return None
    # Keep complete thought blocks, prioritizing the last and earliest context.
    # No invented rationale or gold-derived filler. Budget includes the final answer.
    def completion(parts):
        return '<reasoning>\n'+'\n'.join(parts)+'\n</reasoning>\nFinal answer: '+item['ground_truth']
    full_completion = completion(blocks)
    
    budget = (
        MAX_SEQ_LEN
        - len(encode_ids(item['prompt']))
        - 1
    )
    
    if budget <= 0:
        return None
    
    completion_length = len(
        encode_ids(full_completion)
    )
    
    if completion_length > budget:
        return None
    
    return full_completion

[DATA] Folder: /Users/pritishinde/reasoning-from-scratch/ch06/01_main-chapter-code/rewild_data
[DATA] Train records: 987 | test records: 491


## 2. Tokenizer, split and leakage checks

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id
assert EOS is not None and PAD is not None

def encode_ids(s): return tokenizer.encode(s, add_special_tokens=False)
def normalize(s): return ' '.join(str(s).lower().split())
def components(row):
    return {normalize((q.get('orig_data') or {}).get('question','')) for q in row.get('qls',[]) if (q.get('orig_data') or {}).get('question')}
def fits(row): return len(encode_ids(row['prompt'])) + len(encode_ids(row['completion'])) + 1 <= MAX_SEQ_LEN

def row_id(row): return hashlib.sha256(normalize(row['question']).encode()).hexdigest()[:20]

def stratified_order(rows,seed):
    rng=random.Random(seed); groups=defaultdict(list)
    for row in rows: groups[row['qtype']].append(row)
    for values in groups.values(): rng.shuffle(values)
    keys=sorted(groups); rng.shuffle(keys)
    result=[]
    while any(groups.values()):
        for key in keys:
            if groups[key]: result.append(groups[key].pop())
    return result

processed, skipped, seen = [], Counter(), set()
for raw in train_raw:
    try:
        row=build_item(raw); key=normalize(row['question'])
        if key in seen: skipped['duplicate']+=1; continue
        if len(encode_ids(row['prompt']))+max(EVAL_MAX_NEW_TOKENS,GRPO_MAX_NEW_TOKENS)>MAX_SEQ_LEN:
            skipped['prompt too long for training budget']+=1; continue
        seen.add(key); row['id']=row_id(row); row['components']=sorted(components(raw))
        row['completion']=build_reasoning(raw,row)
        row['source']='dataset_trajectory_thoughts' if row['completion'] else None
        processed.append(row)
    except (KeyError,IndexError,TypeError,ValueError,AttributeError) as exc:
        skipped[str(exc)[:100]]+=1
print('All hybrid candidates:',len(processed),'reasoning available:',sum(bool(x['completion']) for x in processed),'skips:',skipped)
print('Raw train types:',dict(Counter(r.get('qtype','unknown') for r in train_raw)))
trace_rows=[r for r in processed if r['completion'] and fits(r)]
assert len(trace_rows)>=4,'Too few usable trajectory thoughts. Inspect hyb_syn_traj and increase token budgets.'
val_n=min(VAL_EXAMPLES,max(2,len(trace_rows)//10))
val_data=stratified_order(trace_rows,SEED)[:val_n]
val_ids={r['id'] for r in val_data}; val_comps=set().union(*(set(r['components']) for r in val_data))
train_pool=[r for r in processed if r['id'] not in val_ids and not set(r['components'])&val_comps]
# Reserve distinct RL questions; RL does not consume reference thoughts.
rl_n=min(GRPO_STEPS,max(1,len(train_pool)//4))
rl_data=stratified_order(train_pool,SEED+1)[:rl_n]
rl_ids={r['id'] for r in rl_data}
sft_data=stratified_order([r for r in train_pool if r['id'] not in rl_ids and r['completion'] and fits(r)],SEED+2)[:SFT_EXAMPLES]
assert sft_data and rl_data, 'Insufficient disjoint SFT/RL data; inspect counts or reduce validation size.'
assert not ({r['id'] for r in sft_data}&rl_ids)
train_full={normalize(r['input']) for r in train_raw if isinstance(r.get('input'),str)}
train_comps=set().union(*(components(r) for r in train_raw))
test_data=[]
for idx,raw in enumerate(test_raw):
    try: row=build_item(raw)
    except ValueError: continue
    row['id']=row_id(row)+f'-{idx}'
    row['nonoverlap']=normalize(raw['input']) not in train_full and not (components(raw)&train_comps)
    test_data.append(row)
print('Valid official test rows:',len(test_data), 'types:',dict(Counter(r['qtype'] for r in test_data)))
test_data=stratified_order(test_data,SEED+3)
if TEST_EXAMPLES: test_data=test_data[:TEST_EXAMPLES]
assert test_data
for name,rows in [('sft',sft_data),('rl',rl_data),('validation',val_data),('test',test_data)]:
    save_json(name+'_data.json',rows); print(name,len(rows),'types:',dict(Counter(r['qtype'] for r in rows)))
print('Sample SFT:',sft_data[0]['prompt'],sft_data[0]['completion'])

REASONING_SECTION = re.compile(r'(?is)<reasoning>\s*(.*?)\s*</reasoning>')
FINAL_LINE = re.compile(
    r'(?im)^\s*(?:\*\*)?final\s+answer(?:\*\*)?\s*[:=\-]\s*'
    r'(?:the\s+correct\s+(?:statement|option)\s+is\s*)?'
    r'(?:statement|option)?\s*#?\s*(?:\*\*)?(\d+)\b')
ANSWER_LINE = re.compile(r'(?im)^\s*(?:answer|correct\s+(?:statement|option))\s*[:=\-]\s*'
                         r'(?:statement|option)?\s*#?\s*(\d+)\b')
ANSWER_TAG = re.compile(r'(?is)<answer>\s*(?:statement|option)?\s*#?\s*(\d+)\s*</answer>')
ANSWER_SENTENCE = re.compile(r'(?i)\b(?:the\s+)?correct\s+(?:statement|option)\s+is\s*(?:statement|option)?\s*#?\s*(\d+)\b')


def reasoning_body(text):
    matches = list(REASONING_SECTION.finditer(text))
    return matches[-1].group(1) if matches else None


def reasoning_tags_ok(text):
    body = reasoning_body(text)
    return body is not None and len(body.strip()) >= 10


def extract_final_answer(text, n_options=None):
    """Tolerate explicit markdown, N=2 and statement/option labels; never infer from arithmetic."""
    closed=list(REASONING_SECTION.finditer(text))
    region=text[closed[-1].end():] if closed else text
    # Remove markdown markers, not content. Exclude completed reasoning from answer parsing.
    region=region.replace('**','').replace('`','')
    patterns=[
        r'(?im)^\s*(?:[-*]\s*|\d+\.\s*)?final\s+answer\s*[:=\-]\s*(?:the\s+correct\s+(?:statement|option)\s+is\s*)?(?:N\s*=\s*)?(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
        r'(?im)^\s*(?:[-*]\s*)?(?:answer|correct\s+(?:statement|option))\s*[:=\-]\s*(?:N\s*=\s*)?(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
        r'(?is)<answer>\s*(?:statement|option)?\s*#?\s*(\d+)\s*</answer>',
        r'(?i)\b(?:the\s+)?correct\s+(?:statement|option)\s+is\s*(?:statement|option)?\s*#?\s*(\d+)\b(?![.,]\d)',
    ]
    for pattern in patterns:
        matches=list(re.finditer(pattern,region))
        if matches:
            values={int(m.group(1)) for m in matches}
            if len(values)!=1: return None  # contradictory explicit answers
            candidate=values.pop()
            return str(candidate) if n_options is None or 1<=candidate<=n_options else None
    return None

STRICT_FINAL = re.compile(r'(?im)^Final answer:\s*(\d+)\s*$')
def strict_answer(text,n):
    m=list(REASONING_SECTION.finditer(text))
    if len(m)!=1 or not reasoning_tags_ok(text): return None
    tail=text[m[0].end():].strip()
    f=STRICT_FINAL.fullmatch(tail)
    return str(int(f.group(1))) if f and 1<=int(f.group(1))<=n else None
assert extract_final_answer('Final answer: 2',2)=='2'
assert extract_final_answer('4000-1600=2400',2) is None
assert extract_final_answer('**Final answer:** N = 2',2)=='2'
assert extract_final_answer('Final answer: **N=1**',2)=='1'
assert extract_final_answer('Final answer: 1\nFinal answer: 2',2) is None
assert extract_final_answer('<reasoning>Final answer: 1</reasoning>',2) is None
assert extract_final_answer('Final answer: 1.5',2) is None


def score_reasoning(text, example):
    pred=extract_final_answer(text,example['n_options'])
    correct=pred==example['ground_truth']
    format_ok=reasoning_tags_ok(text) and strict_answer(text,example['n_options']) is not None
    # Small format bonus gives a learning signal even if every answer is wrong.
    return dict(reward=float(correct)+0.05*float(format_ok),pred=pred,correct=correct,format_ok=format_ok)

assert score_reasoning('<reasoning>Some reasoning text.</reasoning>\nFinal answer: 1',{'ground_truth':'1','n_options':2})['reward']==1.05
assert score_reasoning('Final answer: 999',sft_data[0])['reward']==0


All hybrid candidates: 987 reasoning available: 987 skips: Counter()
Raw train types: {'gg': 217, 'gf': 226, 'gfx': 173, 'gfr': 195, 'gfrx': 176}
Valid official test rows: 491 types: {'gg': 96, 'gf': 99, 'gfx': 99, 'gfr': 99, 'gfrx': 98}
sft 300 types: {'gg': 60, 'gfx': 60, 'gfr': 60, 'gfrx': 60, 'gf': 60}
rl 100 types: {'gfr': 20, 'gg': 20, 'gf': 20, 'gfx': 20, 'gfrx': 20}
validation 50 types: {'gfrx': 10, 'gfr': 10, 'gfx': 10, 'gf': 10, 'gg': 10}
test 50 types: {'gf': 10, 'gfrx': 10, 'gg': 10, 'gfr': 10, 'gfx': 10}
Sample SFT: ### Problem
Read the context and choose the correct statment.

### Context:
Carol, who receives a steadfast $20 allowance each week, also identifies opportunities to augment her earnings through extra chores, increasing her weekly income by $1.5. As Carol diligently works towards accumulating savings, her culinary preferences also emerge; she typically uses two pieces of bread to craft a regular sandwich, though a more indulgent double meat sandwich demands thr

## 3. Model, padding and generation

In [7]:
from pathlib import Path
import json

DATA_SOURCE = Path("rewild_mps_local_v2")

with open(DATA_SOURCE / "sft_llama_teacher.json") as f:
    train_sft = json.load(f)

with open(DATA_SOURCE / "rl_data.json") as f:
    rl_data = json.load(f)

with open(DATA_SOURCE / "validation_data.json") as f:
    val_data = json.load(f)

with open(DATA_SOURCE / "test_data.json") as f:
    test_data = json.load(f)

assert train_sft, "No teacher-generated examples"

print("Teacher SFT examples:", len(train_sft))
print("GRPO examples:", len(rl_data))
print("Validation examples:", len(val_data))
print("Test examples:", len(test_data))

OUT = Path("rewild_mps_llama_distillation")
OUT.mkdir(parents=True, exist_ok=True)

save_json("sft_final_data.json", train_sft)

Teacher SFT examples: 218
GRPO examples: 100
Validation examples: 50
Test examples: 50


PosixPath('rewild_mps_llama_distillation/sft_final_data.json')

In [8]:
import pandas as pd

summary = []

for name, rows in [
    ("SFT", sft_data),
    ("GRPO", rl_data),
    ("Validation", val_data),
    ("Test", test_data)
]:

    summary.append({
        "Dataset": name,
        "Total examples": len(rows),

        "With reasoning": sum(
            bool(r.get("completion"))
            for r in rows
        ),

        "Average prompt tokens": round(
            sum(
                len(encode_ids(r["prompt"]))
                for r in rows
            ) / len(rows), 2
        ),

        "Reasoning categories": len(
            set(r["qtype"] for r in rows)
        )
    })

display(pd.DataFrame(summary))

,Dataset,Total examples,With reasoning,Average prompt tokens,Reasoning categories
0,SFT,300,300,388.30,5
1,GRPO,100,100,397.87,5
2,Validation,50,50,395.28,5
3,Test,50,0,403.98,5


In [9]:
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=DTYPE,attn_implementation='eager' if device.type=='mps' else 'sdpa').to(device)
model.generation_config.pad_token_id=PAD
model.generation_config.eos_token_id=EOS
model.config.use_cache=False

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows[i]; prompt=encode_ids(row['prompt']); completion=encode_ids(row['completion'])+[EOS]
        assert prompt and len(prompt)+len(completion)<=MAX_SEQ_LEN
        return {'input_ids':prompt+completion,'labels':[-100]*len(prompt)+completion}

def collate_sft(items):
    length=min(MAX_SEQ_LEN,math.ceil(max(len(x['input_ids']) for x in items)/8)*8)
    ids=[]; masks=[]; labels=[]
    for x in items:
        n=len(x['input_ids']); pad=length-n
        ids.append(x['input_ids']+[PAD]*pad); masks.append([1]*n+[0]*pad); labels.append(x['labels']+[-100]*pad)
    return {k:torch.tensor(v,dtype=torch.long) for k,v in [('input_ids',ids),('attention_mask',masks),('labels',labels)]}

sample=collate_sft([SFTDataset(train_sft)[i] for i in range(min(2,len(train_sft)))])
assert (sample['labels'][sample['attention_mask']==0]==-100).all()
assert (sample['labels']!=-100).any(dim=1).all()
for i in range(len(sample['labels'])):
    last=int(sample['attention_mask'][i].sum())-1
    assert sample['labels'][i,last]==EOS
print('Batch shapes:',{k:tuple(v.shape) for k,v in sample.items()})

@torch.no_grad()
def generate_batch(m, prompts, max_new, sample=False):
    m.eval()
    prev=tokenizer.padding_side; tokenizer.padding_side='left'
    try: batch=tokenizer(prompts,return_tensors='pt',padding=True,add_special_tokens=False).to(device)
    finally: tokenizer.padding_side=prev
    width=batch['input_ids'].shape[1]
    assert width+max_new<=m.config.max_position_embeddings, "Prompt exceeds model context; do not silently truncate."
    kwargs=dict(max_new_tokens=max_new,do_sample=sample,use_cache=True,pad_token_id=PAD,eos_token_id=EOS,
                repetition_penalty=1.0, no_repeat_ngram_size=0, min_new_tokens=0,
                forced_bos_token_id=None,forced_eos_token_id=None)
    if sample: kwargs.update(temperature=0.7,top_p=1.0,top_k=0)
    out=m.generate(**batch,**kwargs)
    results=[]
    for i, seq in enumerate(out[:,width:]):
        ids=seq.tolist()
        if EOS in ids: ids=ids[:ids.index(EOS)+1]
        prompt_ids=batch['input_ids'][i][batch['attention_mask'][i].bool()].tolist()
        results.append((prompt_ids,ids,tokenizer.decode(ids,skip_special_tokens=True),len(ids)>=max_new and ids[-1]!=EOS))
    return results

def evaluate(tag, rows, m=None):
    m=model if m is None else m
    records=[]
    for start in range(0,len(rows),EVAL_BATCH):
        chunk=rows[start:start+EVAL_BATCH]
        generations=generate_batch(m,[r['prompt'] for r in chunk],EVAL_MAX_NEW_TOKENS)
        for r,(_,ids,text,truncated) in zip(chunk,generations):
            pred=extract_final_answer(text,r['n_options']); strict=strict_answer(text,r['n_options'])
            metrics=score_reasoning(text,r)
            records.append(dict(id=r['id'],gold=r['ground_truth'],prediction=pred,correct=pred==r['ground_truth'],
                strict_correct=strict==r['ground_truth'],format_ok=strict is not None,reasoning_tags=reasoning_tags_ok(text),
                qtype=r['qtype'],
                nonoverlap=r.get('nonoverlap'),tokens=len(ids),truncated=truncated,response=text))
        print(tag,min(start+len(chunk),len(rows)),'/',len(rows),'accuracy so far',round(sum(x['correct'] for x in records)/len(records),3),flush=True)
    clean=[x for x in records if x['nonoverlap'] is True]
    by_type={}
    for t in sorted({x['qtype'] for x in records}):
        group=[x for x in records if x['qtype']==t]
        by_type[t]={'n':len(group),'accuracy':sum(x['correct'] for x in group)/len(group)}
    mean=lambda k:sum(x[k] for x in records)/len(records)
    report=dict(tag=tag,n=len(records),accuracy=mean('correct'),strict_accuracy=mean('strict_correct'),format_rate=mean('format_ok'),
        extraction_rate=sum(x['prediction'] is not None for x in records)/len(records),truncation_rate=mean('truncated'),
        reasoning_tag_rate=mean('reasoning_tags'),by_type=by_type,
        nonoverlap_n=len(clean),nonoverlap_accuracy=sum(x['correct'] for x in clean)/len(clean) if clean else None,examples=records)
    save_json('eval_'+tag+'.json',report); return report

# Smoke-check actual forward pass before any expensive generation.
with torch.no_grad():
    check=model(**{k:v.to(device) for k,v in sample.items()},use_cache=False).loss
assert torch.isfinite(check),check
print('Base forward loss:',float(check))
base_val=evaluate('base_validation',val_data)

Batch shapes: {'input_ids': (2, 792), 'attention_mask': (2, 792), 'labels': (2, 792)}
Base forward loss: 0.48662668466567993
base_validation 2 / 50 accuracy so far 0.0
base_validation 4 / 50 accuracy so far 0.0
base_validation 6 / 50 accuracy so far 0.0
base_validation 8 / 50 accuracy so far 0.0
base_validation 10 / 50 accuracy so far 0.0
base_validation 12 / 50 accuracy so far 0.0
base_validation 14 / 50 accuracy so far 0.0
base_validation 16 / 50 accuracy so far 0.0
base_validation 18 / 50 accuracy so far 0.0
base_validation 20 / 50 accuracy so far 0.0
base_validation 22 / 50 accuracy so far 0.0
base_validation 24 / 50 accuracy so far 0.0
base_validation 26 / 50 accuracy so far 0.0
base_validation 28 / 50 accuracy so far 0.0
base_validation 30 / 50 accuracy so far 0.0
base_validation 32 / 50 accuracy so far 0.031
base_validation 34 / 50 accuracy so far 0.029
base_validation 36 / 50 accuracy so far 0.028
base_validation 38 / 50 accuracy so far 0.026
base_validation 40 / 50 accuracy so

## 4. SFT — LoRA, masked loss, gradient accumulation and validation selection

In [ ]:
model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.0,bias='none',task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
model.print_trainable_parameters()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
model.enable_input_require_grads()
params=[p for p in model.parameters() if p.requires_grad]
opt=torch.optim.AdamW(params,lr=SFT_LR,weight_decay=0.01,foreach=False)
loader=DataLoader(SFTDataset(train_sft),batch_size=SFT_BATCH,shuffle=True,collate_fn=collate_sft,generator=torch.Generator().manual_seed(SEED))
val_loader=DataLoader(SFTDataset(val_data),batch_size=SFT_BATCH,collate_fn=collate_sft)
updates=math.ceil(len(loader)/GRAD_ACCUM)*SFT_EPOCHS
sched=get_cosine_schedule_with_warmup(opt,num_warmup_steps=max(1,int(.05*updates)) if updates>1 else 0,num_training_steps=updates)

def save_adapter(name):
    path=OUT/name; model.save_pretrained(path); tokenizer.save_pretrained(path); return path

def trainable_state(): return {n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}
@torch.no_grad()
def restore_state(state):
    for n,p in model.named_parameters():
        if n in state: p.copy_(state[n].to(device=p.device,dtype=p.dtype))
@torch.no_grad()
def validation_loss():
    model.eval(); total=0.; count=0
    for batch in val_loader:
        n=int((batch['labels'][:,1:]!=-100).sum()); batch={k:v.to(device) for k,v in batch.items()}
        total+=float(model(**batch,use_cache=False).loss)*n; count+=n
    return total/count

def window_step(window):
    opt.zero_grad(set_to_none=True)
    total=sum(int((b['labels'][:,1:]!=-100).sum()) for b in window); value=0.
    for b in window:
        count=int((b['labels'][:,1:]!=-100).sum()); b={k:v.to(device) for k,v in b.items()}
        loss=model(**b,use_cache=False).loss
        if not torch.isfinite(loss): raise RuntimeError('Nonfinite SFT loss')
        (loss*count/total).backward(); value+=float(loss.detach())*count/total
    norm=torch.nn.utils.clip_grad_norm_(params,1.0,error_if_nonfinite=True)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    return value,float(norm)

sft_history=[]; best_key=None; best_state=None
for epoch in range(1,SFT_EPOCHS+1):
    model.train(); window=[]; losses=[]
    for i,b in enumerate(loader,1):
        window.append(b)
        if len(window)==GRAD_ACCUM or i==len(loader):
            loss,norm=window_step(window); losses.append(loss); window=[]
            print('SFT epoch',epoch,'batch',i,'/',len(loader),'loss',round(loss,4),'grad',round(norm,3),flush=True)
    save_adapter(f'sft_epoch_{epoch}')
    vloss=validation_loss(); report=evaluate(f'sft_validation_{epoch}',val_data)
    key=(report['accuracy'],report['strict_accuracy'],-vloss)
    sft_history.append(dict(epoch=epoch,train_loss=sum(losses)/len(losses),val_loss=vloss,val_accuracy=report['accuracy'],strict_accuracy=report['strict_accuracy']))
    save_json('sft_history.json',sft_history)
    if best_key is None or key>best_key: best_key=key; best_state=trainable_state(); save_adapter('sft_best')
restore_state(best_state); sft_val=evaluate('sft_best_validation',val_data)
del opt,sched; empty_cache()

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
SFT epoch 1 batch 4 / 109 loss 0.504 grad 1.221
SFT epoch 1 batch 8 / 109 loss 0.5773 grad 1.304
SFT epoch 1 batch 12 / 109 loss 0.5835 grad 1.177
SFT epoch 1 batch 16 / 109 loss 0.5569 grad 1.001
SFT epoch 1 batch 20 / 109 loss 0.4152 grad 0.767
SFT epoch 1 batch 24 / 109 loss 0.5457 grad 1.012
SFT epoch 1 batch 28 / 109 loss 0.5588 grad 0.877
SFT epoch 1 batch 32 / 109 loss 0.5036 grad 0.924
SFT epoch 1 batch 36 / 109 loss 0.4438 grad 0.82
SFT epoch 1 batch 40 / 109 loss 0.3966 grad 0.916
SFT epoch 1 batch 44 / 109 loss 0.4679 grad 1.047
SFT epoch 1 batch 48 / 109 loss 0.5224 grad 1.06
SFT epoch 1 batch 52 / 109 loss 0.5059 grad 1.025
SFT epoch 1 batch 56 / 109 loss 0.3971 grad 0.979
SFT epoch 1 batch 60 / 109 loss 0.3732 grad 0.86
SFT epoch 1 batch 64 / 109 loss 0.5324 grad 0.907
SFT epoch 1 batch 68 / 109 loss 0.565 grad 1.147
SFT epoch 1 batch 72 / 109 loss 0.482 grad 0.916
SFT epoch 1 batch 76 / 109 los

## 5. GRPO-style clipped policy optimization with fixed SFT reference

In [ ]:
# Reuse one frozen base model; store the fixed SFT adapter on CPU.
reference_state=trainable_state()
# Keep gradients on the live adapter only; eval() disables dropout, not autograd.
model.eval()
rl_opt=torch.optim.AdamW(params,lr=GRPO_LR,weight_decay=0,foreach=False)

def token_logps(m,prompt_ids,completion_ids):
    ids=torch.tensor([prompt_ids+completion_ids],device=device,dtype=torch.long)
    p=len(prompt_ids); c=len(completion_ids)
    assert p>0 and c>0
    # Qwen supports logits_to_keep: avoid projecting unused prompt tokens.
    logits=m(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,logits_to_keep=c+1).logits[0,:-1].float()
    target=ids[0,p:]
    assert logits.shape[0]==len(target)
    return -F.cross_entropy(logits,target,reduction='none')

rl_history=[]; rl_best_key=(sft_val['accuracy'],sft_val['strict_accuracy']); rl_best_state=None
for step in range(1,GRPO_STEPS+1):
    # Cycle shuffled training-only RL pool if it is smaller than requested groups.
    if (step-1)%len(rl_data)==0:
        rl_order=list(rl_data); random.Random(SEED+step).shuffle(rl_order)
    row=rl_order[(step-1)%len(rl_data)]
    generated=[generate_batch(model,[row['prompt']],GRPO_MAX_NEW_TOKENS,sample=True)[0] for _ in range(GRPO_ROLLOUTS)]
    metrics=[score_reasoning(t,row) for _,_,t,_ in generated]
    rewards=torch.tensor([m['reward'] for m in metrics],device=device,dtype=torch.float32)
    log=dict(step=step,id=row['id'],mean_reward=float(rewards.mean()),updated=False,
             rollouts=[dict(response=t,truncated=tr,tokens=len(c),**m) for (_,c,t,tr),m in zip(generated,metrics)])
    if rewards.std(unbiased=False).item()>1e-7:
        adv=(rewards-rewards.mean())/(rewards.std(unbiased=False)+1e-4)
        with torch.no_grad():
            old=[token_logps(model,p,c).detach() for p,c,_,_ in generated]
            live_state=trainable_state()
            try:
                restore_state(reference_state)
                reference=[token_logps(model,p,c).detach() for p,c,_,_ in generated]
            finally:
                restore_state(live_state)
            del live_state
        for _ in range(GRPO_OPT_EPOCHS):
            rl_opt.zero_grad(set_to_none=True); loss_sum=0.; kl_sum=0.
            for j,(p,c,_,_) in enumerate(generated):
                current=token_logps(model,p,c)
                ratio=torch.exp((current-old[j]).clamp(-20,20))
                policy=-torch.minimum(ratio*adv[j],ratio.clamp(1-GRPO_CLIP_EPS,1+GRPO_CLIP_EPS)*adv[j])
                delta=(reference[j]-current).clamp(-10,10)
                kl=torch.expm1(delta)-delta
                loss=(policy+GRPO_KL_BETA*kl).mean()/GRPO_ROLLOUTS
                if not torch.isfinite(loss): raise RuntimeError('Nonfinite GRPO loss')
                loss.backward(); loss_sum+=float(loss.detach()); kl_sum+=float(kl.detach().mean())/GRPO_ROLLOUTS
            torch.nn.utils.clip_grad_norm_(params,1.0,error_if_nonfinite=True); rl_opt.step()
        log.update(updated=True,loss=loss_sum,kl=kl_sum)
    rl_history.append(log); save_json('grpo_history.json',rl_history)
    print('GRPO',step,'/',GRPO_STEPS,'reward',round(log['mean_reward'],3),'updated',log['updated'],flush=True)
    if step%GRPO_VAL_EVERY==0 or step==GRPO_STEPS:
        save_adapter(f'grpo_step_{step}')
        report=evaluate(f'grpo_validation_{step}',val_data)
        key=(report['accuracy'],report['strict_accuracy'])
        if key>rl_best_key:
            rl_best_key=key; rl_best_state=trainable_state(); save_adapter('grpo_best')
last_state=trainable_state(); save_adapter('grpo_last')
print('Updated groups:',sum(x['updated'] for x in rl_history),'/',len(rl_history))
print('Validation-selected RL improvement:',rl_best_state is not None)
del reference_state,rl_opt; empty_cache()

## 6. Final untouched official test comparison

In [ ]:
from peft import PeftModel
# Disable adapter for pretrained baseline; base was frozen throughout.
with model.disable_adapter():
    base_eval=evaluate('base_test',test_data)
restore_state(best_state)
sft_eval=evaluate('sft_test',test_data)
restore_state(last_state)
rl_eval=evaluate('grpo_last_test',test_data)
reports={'base':base_eval,'sft':sft_eval,'sft_grpo_last':rl_eval}
if rl_best_state is not None:
    restore_state(rl_best_state)
    selected=evaluate('grpo_selected_test',test_data) if rl_best_state is not last_state else rl_eval
    reports['sft_grpo_selected']=selected
    save_adapter('deployment_selected')
else:
    restore_state(best_state); save_adapter('deployment_selected')

def paired(a,b):
    A={x['id']:x['correct'] for x in a['examples']}; B={x['id']:x['correct'] for x in b['examples']}
    assert A.keys()==B.keys()
    gains=sum(not A[k] and B[k] for k in A); losses=sum(A[k] and not B[k] for k in A)
    return {'wrong_to_right':gains,'right_to_wrong':losses,'delta_accuracy':(gains-losses)/len(A)}
print('MODEL                         ACC     STRICT   EXTRACT   CLEAN ACC')
for name,r in reports.items():
    clean='N/A' if r['nonoverlap_accuracy'] is None else f"{r['nonoverlap_accuracy']:.1%}"
    print(f"{name:27s} {r['accuracy']:7.1%} {r['strict_accuracy']:9.1%} {r['extraction_rate']:9.1%} {clean}")
summary=dict(profile=PROFILE,model=MODEL_ID,seed=SEED,n_sft=len(train_sft),n_unique_sft=len(sft_data),n_rl_unique=len(rl_data),n_val=len(val_data),n_test=len(test_data),grpo_groups=GRPO_STEPS,grpo_updated=sum(x['updated'] for x in rl_history),selected='grpo' if rl_best_state is not None else 'sft',config=dict(sft_epochs=SFT_EPOCHS,sft_lr=SFT_LR,grpo_lr=GRPO_LR,rollouts=GRPO_ROLLOUTS,kl_beta=GRPO_KL_BETA,max_seq_len=MAX_SEQ_LEN,eval_max_new=EVAL_MAX_NEW_TOKENS,grpo_max_new=GRPO_MAX_NEW_TOKENS,batch=SFT_BATCH,grad_accum=GRAD_ACCUM),reports=reports,paired={k:paired(base_eval,v) for k,v in reports.items() if k!='base'})
print('Final report:',save_json('final_comparison.json',summary))
print('SFT -> GRPO:',paired(sft_eval,rl_eval))
for name,report in reports.items(): print(name,'per-type accuracy:',report['by_type'])
print('Selected adapter:',OUT/'deployment_selected')